In [1]:
import os, sys
sys.path.append(os.path.abspath("..")) 
from src.state_schema import SupervisorState
from src.supervisor_graph import supervisor_node
from src.complaint_graph import complaint_flow
from src.inquiry_graph import inquiry_flow
from src.retention_graph import retention_flow
from src.conversation_graph import greeting_flow, clarification_flow
from rich import print as rprint

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

In [3]:
def build_graph():

    graph = StateGraph(SupervisorState)

    graph.add_node("supervisor",supervisor_node)
    graph.add_node("greeting_flow",greeting_flow)
    graph.add_node("clarification_flow", clarification_flow)
    graph.add_node("complaint_flow", complaint_flow)
    graph.add_node("retention_flow", retention_flow)
    graph.add_node("inquiry_flow", inquiry_flow)

    graph.add_edge(START, "supervisor")
    graph.add_edge("greeting_flow", END)
    graph.add_edge("clarification_flow", END)

    memory = MemorySaver()
    return graph.compile(checkpointer=memory)

graph = build_graph()


In [4]:
def agent(state:SupervisorState)->SupervisorState:
    config = {"configurable": {"thread_id": "support_session_1"}}
    return graph.invoke(state,config)

In [5]:
test_cases = [
    # --- FLOW 1: GREETINGS & CHITCHAT ---
    {"user_input": "Hello! Is there anyone available to help me with a few questions?"},
    {"user_input": "Good morning, I hope you're having a productive day."},

    # --- FLOW 2: TECHNICAL COMPLAINTS (Complaint Graph) ---
    {"user_input": "My home internet has been dropping every 20 minutes since the rain started."},
    {"user_input": "The screen on my new device is flickering and I can't see the menu."},
    {"user_input": "I've already tried restarting the router, but the red light is still blinking."},

    # --- FLOW 3: ACCOUNT & RETENTION (Retention Graph) ---
    {"user_input": "I want to cancel my subscription effective immediately; it's too expensive."},
    {"user_input": "My contract is up next month and I'm looking at switching to a different provider."},
    {"user_input": "I was promised a discount on my last bill that never showed up. I'm very upset."},

    # --- FLOW 4: GENERAL INQUIRIES (Inquiry Graph) ---
    {"user_input": "How do I set up international roaming for my upcoming trip to Europe?"},
    {"user_input": "What are your store hours for the downtown St. Louis location?"},
    {"user_input": "Can you explain the difference between the Basic and Pro service plans?"},

    # --- FLOW 5: AMBIGUOUS (Clarification Node) ---
    {"user_input": "I'm having a really hard time with this service and I need a solution."},
    {"user_input": "This is the third time I'm reaching out about my order status."},
    {"user_input": "I'm not sure if I'm in the right place, but I need help with my account."},

    # --- MULTI-TURN / STICKY FLOW TESTS (Requires same thread_id) ---
    {"user_input": "My phone won't turn on."}, # Turn 1: Should enter Complaint
    {"user_input": "Yes, I held the power button for 10 seconds like you said."}, # Turn 2: Should stay in Complaint
    {"user_input": "Wait, actually, how much would it cost just to upgrade to a new phone instead?"} # Turn 3: Should pivot or reset
]

In [10]:
res = agent({"user_input":
             "Mar 10,2025."})

active_flow --> None
active_flow: None, margin: 0.10035824775695801, label: greeting_flow


In [6]:
res = agent(test_cases[3])

active_flow --> []
active_flow: [], margin: 0.26115162670612335, label: complaint_flow


In [9]:
for msg in res['messages']:
    rprint(msg.content)

It looks like you're trying to report a product issue. However, I'm missing some information to better assist you. 
Could you please provide your purchase date for the device? The purchase date can be your original purchase date or
the date when you received the device. This information will help me provide a more accurate and helpful response. 

You can provide the purchase date in the format 'YYYY-MM-DD' or describe it in a way that I can understand (e.g., 
'last month', 'in January', etc.).

Ticket BV905D20 has been created.

In [16]:
res

{'messages': [HumanMessage(content="I understand you're still having issues with the router. Can you please tell me what you're trying to accomplish with your router?", additional_kwargs={}, response_metadata={}, id='5414a30d-4624-408b-b1c1-4c88606359be'),
  HumanMessage(content="I'd be happy to help with the issue, could you please tell me what's not working correctly with the router?", additional_kwargs={}, response_metadata={}, id='43a473dd-56a7-4826-8e6c-4cad633ac9ed'),
  HumanMessage(content='"I\'m so sorry to hear that our service isn\'t meeting your budget. That really surprises me, given your loyalty to our community. As a token of our appreciation, I\'d like to offer you a full year of premium access for free. If you\'re willing to give us another chance, I\'d be happy to set that up for you. Please let me know if there\'s anything else I can do to make it right."', additional_kwargs={}, response_metadata={}, id='326b2cf5-8ed4-48df-a441-94deba799658'),
  HumanMessage(content="